# Train Shot Type Model

This notebook loads the synthetic snooker dataset, trains a DecisionTreeClassifier to predict `shot_type`, evaluates it, and saves the trained model.

In [3]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier


current_dir = Path.cwd().resolve()
if (current_dir / "data").exists():
    base_dir = current_dir
elif (current_dir.parent / "data").exists():
    base_dir = current_dir.parent
else:
    base_dir = current_dir

data_path = base_dir / "data" / "shots.csv"
model_path = base_dir / "models" / "shot_model.pkl"


df = pd.read_csv(data_path)
df.head()

,cue_x,cue_y,target_x,target_y,pocket_x,pocket_y,angle_to_pocket,cut_angle,distance_cue_to_target,distance_target_to_pocket,num_balls_in_path,ball_colour,is_snookered,shot_score,is_recommended,shot_type
0,2.198035,1.023434,2.103795,0.492918,1.434894,1.446288,125.054144,45.127055,0.538822,1.164621,1,yellow,False,19.114267,False,medium_cut
1,1.246415,1.009958,2.828090,0.597843,0.020444,0.018689,-168.344640,26.259388,1.634483,2.866757,0,black,True,8.971693,False,thin_cut
2,2.438418,0.287827,0.116376,1.398273,-0.015167,0.029246,-95.488390,69.930350,2.573902,1.375333,3,red,False,0.000000,False,heavy_cut
3,1.980525,0.052051,0.695571,0.119556,-0.027044,0.035820,-173.390150,9.617114,1.286726,0.727450,0,green,False,61.038492,False,straight
4,0.267464,0.431395,1.697584,0.158545,-0.022413,1.425134,143.632419,25.566006,1.455916,2.136033,0,brown,False,9.572373,False,thin_cut


In [4]:
feature_columns = [
    "cue_x",
    "cue_y",
    "target_x",
    "target_y",
    "pocket_x",
    "pocket_y",
    "angle_to_pocket",
    "cut_angle",
    "distance_cue_to_target",
    "distance_target_to_pocket",
    "num_balls_in_path",
    "ball_colour",
    "is_snookered",
]

X = df[feature_columns]
y = df["shot_type"]

numeric_features = [column for column in feature_columns if column != "ball_colour"]
categorical_features = ["ball_colour"]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42)),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 score: {f1:.4f}")

model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_path)
print(f"Saved model to {model_path}")

Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 score: 1.0000
Saved model to /home/half/Documents/6th Assignment/smart-snooker/models/shot_model.pkl


In [ ]:
from sklearn.ensemble import RandomForestClassifier


rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=200, random_state=42)),
    ]
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, average="weighted", zero_division=0)
rf_recall = recall_score(y_test, rf_pred, average="weighted", zero_division=0)
rf_f1 = f1_score(y_test, rf_pred, average="weighted", zero_division=0)

comparison = pd.DataFrame(
    {
        "Decision Tree": [accuracy, precision, recall, f1],
        "Random Forest": [rf_accuracy, rf_precision, rf_recall, rf_f1],
    },
    index=["Accuracy", "Precision", "Recall", "F1 Score"],
)

print("Decision Tree vs Random Forest")
print(comparison.round(4).to_string())